In [3]:
# Cell 1: 导入依赖 & 配置参数
import re
import psycopg2
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ---------- 数据库连接配置 ----------
DB_CONFIG = {
    "dbname": "lawApp_LangGraph",
    "user": "my_pgsql",
    "password": "PGsql123123",
    "host": "rm-cn-rfd4pkzs1000112o.rwlb.rds.aliyuncs.com",
    "port": 5432,
}

# ---------- 嵌入模型 ----------
MODEL_NAME = "BAAI/bge-large-zh-v1.5"
model = SentenceTransformer(MODEL_NAME)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
import re
from pathlib import Path


def parse_law_from_file(file_path: str):
    """
    从法律文本文件中解析法条。
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict] 每个元素包含 chapter, article_number, content
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ---------- 匹配章标题 ----------
    chapter_pattern = re.compile(r"第([一二三四五六七八九十百千零]+)章\s*(.*)")
    chapters = list(chapter_pattern.finditer(text))

    if not chapters:
        sections = [(None, "", 0, len(text))]
    else:
        sections = []
        for i, m in enumerate(chapters):
            chapter_name = f"第{m.group(1)}章 {m.group(2).strip()}"
            start = m.start()
            end = chapters[i + 1].start() if i + 1 < len(chapters) else len(text)
            sections.append((chapter_name, start, end))

    # ---------- 匹配条文 ----------
    article_pattern = re.compile(
        r"第([一二三四五六七八九十百千零]+)条\s*(.*?)(?=第[一二三四五六七八九十百千零]+条|第[一二三四五六七八九十百千零]+章|$)",
        re.DOTALL,
    )

    # ---------- 过滤施行日期条款 ----------
    _DATE_CLAUSE_RE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def is_effective_date_clause(content: str) -> bool:
        c = content.strip()
        return len(c) < 80 and bool(_DATE_CLAUSE_RE.search(c))

    articles = []
    for section_name, sec_start, sec_end in sections:
        section_text = text[sec_start:sec_end]
        for m in article_pattern.finditer(section_text):
            article_num = m.group(1)
            raw_content = m.group(2).strip()
            if is_effective_date_clause(raw_content):
                continue
            articles.append(
                {
                    "chapter": section_name or "",
                    "article_number": f"第{article_num}条",
                    "content": raw_content,
                }
            )

    return law_title, articles

In [2]:
# Cell 3: 数据库建表（首次运行执行一次）
def create_table_if_not_exists():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS marriage_law (
            id SERIAL PRIMARY KEY,
            law_title TEXT NOT NULL,
            chapter TEXT,
            article_number TEXT NOT NULL,
            content TEXT NOT NULL,
            embedding VECTOR(1024),
            UNIQUE(law_title, article_number)
        );
        -- 如果需要索引可后续创建
        -- CREATE INDEX ON statutes USING hnsw (embedding vector_cosine_ops) WHERE embedding IS NOT NULL;
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("表已就绪。")


create_table_if_not_exists()


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd6 in position 61: invalid continuation byte

In [ ]:
# Cell 4: 向量化并插入数据库
def insert_articles(law_title: str, articles: list[dict]):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    sql = """
        INSERT INTO marriage_law (law_title, chapter, article_number, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (law_title, article_number) DO UPDATE
        SET content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            chapter = EXCLUDED.chapter
    """
    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
            ),
        )
    conn.commit()
    cur.close()
    conn.close()
    print(f"成功插入/更新 {len(articles)} 条记录。")

In [ ]:
# Cell 5: 假数据测试（使用样例文本）
sample_text = """第六章　附　　则

第二十五条　中华人民共和国驻外使（领）馆可以依照本条例的有关规定，为男女双方均居住于驻在国的中国公民办理婚姻登记。
第二十六条　男女双方均非内地居民的中国公民在内地办理婚姻登记的具体办法，由国务院民政部门另行制定。
第二十七条　本条例规定的婚姻登记证由国务院民政部门规定式样并监制。
第二十八条　本条例自2025年5月10日起施行。"""

# 将样例文本写入临时文件
sample_file = "sample_law.txt"
with open(sample_file, "w", encoding="utf-8") as f:
    f.write(sample_text)

# 解析
law_title, articles = parse_law_from_file(sample_file)
print(f"法律名称：{law_title}")
for a in articles:
    print(f"{a['article_number']} [{a['chapter']}] {a['content'][:30]}...")

# 插入数据库（第二十八条会被自动过滤）
insert_articles(law_title, articles)

# 验证：查询一条记录
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute(
    "SELECT law_title, article_number, content FROM marriage_law WHERE law_title=%s LIMIT 1;",
    (law_title,),
)
row = cur.fetchone()
print(f"数据库验证：{row}")
cur.close()
conn.close()